# 03 · Build the DuckDB warehouse and answer the business questions in SQL

This notebook only *runs* SQL — all the logic lives in the `sql/` folder so it can be read
(and run) on its own:

| File | Does |
|---|---|
| `sql/01_load_raw.sql` | Copies the source files into the warehouse unchanged |
| `sql/02_build_tables.sql` | Star schema + summary tables + renewal / provision analysis |
| `sql/03_quality_checks.sql` | 18 data-quality checks — the build **stops** if an error-level check fails |
| `sql/04_analysis_queries.sql` | The business questions (Q1–Q11) |
| `sql/05_export_powerbi.sql` | CSV exports for Power BI |

**DuckDB** is a database stored in a single file (`data/warehouse.duckdb`) — no server to
install. The SQL is standard and would run on Postgres or Snowflake with minor changes.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb
import pandas as pd
import yaml

from src.logger import get_logger

log = get_logger("03_build_warehouse")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

con = duckdb.connect("data/warehouse.duckdb")


def run_sql_file(path):
    """Run every statement in a .sql file."""
    con.execute(open(path, encoding="utf-8").read())
    log.info(f"Ran {path}")

## 1. Load the config values the SQL needs
Spreads, LGDs and thresholds come from `config/pipeline.yml`, loaded into a small table
`raw.assumptions`. That way **no number is typed directly into the SQL**.

In [2]:
config = yaml.safe_load(open("config/pipeline.yml"))

assumptions = pd.DataFrame(
    [
        ("fixed_spread_over_goc_5yr", config["risk"]["mortgage"]["fixed_spread_over_goc_5yr"]),
        ("fixed_rate_floor", config["risk"]["mortgage"]["fixed_rate_floor"]),
        ("rate_increase_per_pp", config["hazard_drivers"]["rate_increase_per_pp"]),
        ("severe_payment_shock_pct", config["analysis"]["severe_payment_shock_pct"]),
    ]
    + [(f"lgd_{product}", settings["lgd"]) for product, settings in config["risk"].items()],
    columns=["name", "value"],
)

con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE OR REPLACE TABLE raw.assumptions AS SELECT * FROM assumptions")
assumptions

,name,value
0,fixed_spread_over_goc_5yr,1.40
1,fixed_rate_floor,1.35
2,rate_increase_per_pp,0.25
3,severe_payment_shock_pct,25.00
4,lgd_mortgage,0.10
5,lgd_heloc,0.25
6,lgd_auto,0.45
7,lgd_card,0.85


## 2. Load raw data and build the tables

In [3]:
run_sql_file("sql/01_load_raw.sql")
run_sql_file("sql/02_build_tables.sql")

tables = con.sql("""
    SELECT schema_name, table_name, estimated_size AS row_count
    FROM duckdb_tables()
    ORDER BY schema_name, table_name
""").df()
tables

2026-10-17 17:34:28,940 | 03_build_warehouse | INFO | Ran sql/01_load_raw.sql


2026-10-17 17:34:29,399 | 03_build_warehouse | INFO | Ran sql/02_build_tables.sql


,schema_name,table_name,row_count
0,mart,agg_portfolio_monthly,8361
1,mart,agg_vintage,46
2,mart,dim_credit_band,5
3,mart,dim_loan,50000
4,mart,dim_month,156
5,mart,dim_product,4
6,mart,dim_province,10
7,mart,fact_loan_month,1688432
8,mart,fact_renewal,6716
9,mart,loan_outcome,50000


## 3. Data quality gate
Each check counts rows that **break** a rule. If any `error` check fails, this cell raises an
error and the rest of the notebook doesn't run — bad data never reaches the dashboard.

In [4]:
run_sql_file("sql/03_quality_checks.sql")
checks = con.sql("SELECT dimension, check_name, severity, failed_rows, result FROM quality.check_results").df()

failed_errors = checks[(checks["result"] == "FAIL") & (checks["severity"] == "error")]
if len(failed_errors) > 0:
    raise ValueError(f"{len(failed_errors)} data quality checks failed:\n{failed_errors}")

log.info(f"Data quality: {(checks['result'] == 'PASS').sum()} of {len(checks)} checks passed")
checks

2026-10-17 17:34:29,555 | 03_build_warehouse | INFO | Ran sql/03_quality_checks.sql


2026-10-17 17:34:29,559 | 03_build_warehouse | INFO | Data quality: 18 of 18 checks passed


,dimension,check_name,severity,failed_rows,result
0,completeness,"Every loan has a product, province and credit ...",error,0,PASS
1,completeness,Every month in the data has all four real rates,error,0,PASS
2,completeness,Every renewing mortgage has old and new payment,error,0,PASS
3,uniqueness,loan_id is unique in dim_loan,error,0,PASS
4,uniqueness,One row per loan per month in fact_loan_month,error,0,PASS
5,uniqueness,One row per month in dim_month,error,0,PASS
6,validity,Credit scores between 300 and 900,error,0,PASS
7,validity,Balances are not negative,error,0,PASS
8,validity,Status is one of the six allowed values,error,0,PASS
9,validity,Interest rates between 0% and 30%,error,0,PASS


## 4. The business questions
Each query lives in `sql/04_analysis_queries.sql`. `ask("Q1")` runs one of them.

In [5]:
# Split the file into its queries and file them under "Q1", "Q2", ...
queries = {}
for statement in open("sql/04_analysis_queries.sql", encoding="utf-8").read().split(";"):
    for line in statement.splitlines():
        if line.startswith("-- Q"):
            key = line.split(".")[0].replace("-- ", "")   # "-- Q1. How big..." -> "Q1"
            queries[key] = (line.replace("-- ", ""), statement)
            break


def ask(key):
    title, sql = queries[key]
    print(title)
    return con.sql(sql).df()


print(f"{len(queries)} questions loaded: {', '.join(queries)}")

12 questions loaded: Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q8, Q9, Q10, Q11, Q12


### Q12 · The headline numbers first
These are the five KPI cards on the dashboard. Use this row to check your Power BI cards.

In [6]:
ask("Q12")

Q12. Headline numbers (the KPI cards on the dashboard, and the README summary)


,total_exposure_bn,delinquency_rate_pct,impaired_rate_pct,renewing_mortgages,renewing_balance_bn,weighted_avg_shock_pct,at_risk_balance_m,added_annual_payments_m,incremental_provision_cad,borrowers_still_to_call
0,8.41,0.53,0.09,6716,2.7,15.2,244.9,28.3,193428.0,2797


### Q1 · The renewal wave, year by year

In [7]:
ask("Q1")

Q1. How big is the renewal wave, and how hard does it hit each year?


,renewal_year,renewing_loans,already_renewed,renewing_balance_m,weighted_avg_shock_pct,added_annual_payments_m,at_risk_balance_m
0,2025,2189,2189,852.8,19.8,11.22,164.9
1,2026,2250,1716,900.8,20.2,12.46,79.9
2,2027,2277,0,950.0,6.3,4.61,0.0


**Reading it:** 2025 and 2026 renewals face a ~20% higher payment (balance-weighted). 2027 is
much milder (~6%) because those terms started in 2022, after rates had already risen.
All 2025 renewals have already happened by Sep 2026; most of 2026 has; none of 2027 has — those
are the borrowers the bank can still reach before the shock.

### Q2 · How big is the shock?

In [8]:
ask("Q2")

Q2. What does the shock look like? Loans and balance by shock band


,shock_band,loans,balance_m,pct_of_renewing_balance,avg_monthly_increase_cad
0,0-10%,1707,726.5,26.9,103.0
1,10-25%,4445,1732.3,64.1,420.0
2,Over 25%,564,244.9,9.1,561.0


### Q3 · What actually drives the shock: *when* the term started

In [9]:
ask("Q3")

Q3. Which year did the current term start? (this is what really drives the shock)


,term_started,loans,median_old_rate,median_new_rate,median_shock_pct,median_monthly_increase_cad
0,2020,2189,1.82,4.20,19.1,405.0
1,2021,2250,2.28,4.56,20.2,450.0
2,2022,2277,4.38,4.94,4.2,120.0


**Reading it:** a term locked in 2020 at a median 1.82% renews at 4.20%; 2021 at 2.28% → 4.56%.
Terms locked in 2022 started at 4.38%, so they barely move. **Timing, not borrower quality,
decides who gets shocked.**

### Q4 · Provinces

In [10]:
ask("Q4")

Q4. Which provinces carry the most renewal risk?


,province_name,renewing_loans,renewing_balance_m,weighted_avg_shock_pct,incremental_provision_cad,pct_of_total_provision
0,Ontario,2522,1029.7,15.2,73684.0,38.1
1,Quebec,1517,604.0,14.9,42247.0,21.8
2,British Columbia,965,386.2,15.3,27735.0,14.3
3,Alberta,810,328.8,15.3,23770.0,12.3
4,Manitoba,236,93.0,15.2,6391.0,3.3
5,Nova Scotia,202,84.3,15.7,6221.0,3.2
6,Saskatchewan,202,76.9,15.1,5778.0,3.0
7,New Brunswick,135,54.1,15.6,3756.0,1.9
8,Newfoundland and Labrador,96,34.7,14.7,2706.0,1.4
9,Prince Edward Island,31,12.0,15.7,1139.0,0.6


**Reading it (honestly):** the shock is about 15% in every province, and each province's share
of the provision tracks its size (Ontario ≈ 38%). That's expected: in this *simulated* book,
provinces are assigned at random and there is no regional economy in the model, so regional
differences are only about **where the balances are**, not regional stress. Adding real
provincial unemployment (Statistics Canada) would be the next step to make this question richer.

### Q5 · Does the shock land on weaker borrowers?
**Lift** = a band's share of the high-shock group ÷ its share of all renewals.
Lift > 1 means over-represented.

In [11]:
ask("Q5")

Q5. Does payment shock fall on weaker borrowers? Compare each credit band's share of


,credit_band,all_renewals,high_shock,pct_of_renewals,pct_of_high_shock,lift
0,Below 600,302,84,4.5,3.9,0.86
1,600-659,1015,313,15.1,14.4,0.96
2,660-719,2030,665,30.2,30.7,1.01
3,720-779,1997,657,29.7,30.3,1.02
4,780+,1372,449,20.4,20.7,1.01


**Reading it:** every band's lift is close to 1 (0.86–1.02). The shock is spread evenly across
credit quality — it depends on when the rate was locked, not on who the borrower is. The *risk*
is where the shock meets a weak credit score, which is what the provision (Q6) and the call list
(Q10) pick up.

### Q6 · Extra provision by credit band

In [12]:
ask("Q6")

Q6. Where should the extra provision be held? By credit band.


,credit_band,renewing_loans,base_pd_12m_pct,stressed_pd_12m_pct,expected_loss_before_cad,expected_loss_after_cad,incremental_provision_cad
0,Below 600,302,0.550,0.788,68410.0,97534.0,29124.0
1,600-659,1015,0.290,0.415,119796.0,170618.0,50823.0
2,660-719,2030,0.191,0.275,154805.0,222279.0,67473.0
3,720-779,1997,0.085,0.122,67311.0,96885.0,29574.0
4,780+,1372,0.068,0.098,37839.0,54274.0,16434.0


**Reading it:** the mortgage default probability falls steadily with credit score (Q8), so
"Below 600" borrowers carry the most extra provision *per loan*. The 660–719 band holds the
most provision *overall* simply because it has the most loans.

**Caveat:** each band has only 6–25 mortgage defaults, so these PDs carry real sampling noise.
A bank would smooth them with a model — notebook 05 fits one.

### Q7 · Portfolio health today, by product

In [13]:
ask("Q7")

Q7. Headline portfolio KPIs in the latest month, by product


,product_name,open_loans,exposure_m,delinquency_rate_pct,impaired_rate_pct
0,Mortgage,17378,7434.2,0.47,0.07
1,HELOC,4863,655.3,0.80,0.18
2,Auto loan,6240,142.3,1.11,0.21
3,Credit card,16993,175.4,1.48,0.51


### Q8 · Default rates by product and credit band
Sanity check: secured lending should default least, unsecured most
(mortgage < HELOC < auto < card), and lower credit bands should default more.

In [14]:
ask("Q8")

Q8. Default rates by product and credit band (the PD table behind the provision)


,product_name,credit_band,loans_in_cohort,defaulted_loans,default_rate_42m_pct,pd_12m_pct
0,Mortgage,Below 600,523,10,1.91,0.550
1,Mortgage,600-659,1880,19,1.01,0.290
2,Mortgage,660-719,3758,25,0.67,0.191
3,Mortgage,720-779,3722,11,0.30,0.085
4,Mortgage,780+,2526,6,0.24,0.068
5,HELOC,Below 600,177,4,2.26,0.651
6,HELOC,600-659,557,14,2.51,0.725
7,HELOC,660-719,1026,14,1.36,0.392
8,HELOC,720-779,1049,12,1.14,0.328
9,HELOC,780+,680,5,0.74,0.211


### Q9 · Did the 2023 rate spike show up in HELOCs?

In [15]:
ask("Q9")

Q9. Did rising rates show up in HELOCs (their rate floats with prime)? HELOC delinquency vs prime, by quarter


,year,quarter,avg_prime_rate,heloc_delinquency_pct
0,2023,2,6.76,0.65
1,2023,3,7.16,0.79
2,2023,4,7.20,0.97
3,2024,1,7.20,0.96
4,2024,2,7.14,0.68
5,2024,3,6.70,0.60
6,2024,4,6.00,0.53
7,2025,1,5.24,0.52
8,2025,2,4.95,0.69
9,2025,3,4.93,0.64


**Reading it:** HELOC rates float with prime. As prime climbed to 7.2% through 2023, HELOC
delinquency rose from 0.65% to a peak of 0.97% (Q4 2023), then eased to ~0.5–0.7% as prime fell
back to 4.45%. This is the real rate path feeding through the simulation.

### Q10 · The call list: who to contact first
Only borrowers who **haven't renewed yet** and **haven't defaulted**, ranked by how much extra
loss their renewal adds (sub-600 scores + a big rate jump rise to the top).

In [16]:
ask("Q10")

Q10. Who do we call first? Top 15 borrowers who haven't renewed yet


,outreach_priority,loan_id,province_name,credit_score,renewal_month,old_rate,new_rate,monthly_increase_cad,payment_shock_pct,incremental_provision_cad
0,1,30258,Alberta,586,2026-10-01,2.83,5.14,875.0,22.26,229.0
1,2,39966,Quebec,595,2026-12-01,2.86,5.13,766.0,21.80,200.0
2,3,4495,Ontario,533,2027-02-01,3.37,5.22,744.0,17.21,192.0
3,4,15781,Prince Edward Island,516,2026-12-01,2.98,5.24,701.0,16.55,191.0
4,5,43834,Alberta,586,2026-11-01,3.04,5.14,726.0,19.99,188.0
5,6,13416,Ontario,563,2027-01-01,3.18,5.17,713.0,18.77,185.0
6,7,38756,Ontario,585,2026-10-01,2.83,5.14,640.0,22.26,167.0
7,8,20372,Prince Edward Island,583,2026-10-01,2.83,5.14,602.0,16.94,165.0
8,9,15607,Saskatchewan,555,2026-11-01,3.08,5.19,593.0,15.25,162.0
9,10,42355,Ontario,592,2026-12-01,2.86,5.13,556.0,16.60,153.0


### Q11 · Data quality summary

In [17]:
ask("Q11")

Q11. Data quality summary


,dimension,checks,passed,failed
0,completeness,3,3,0
1,consistency,4,4,0
2,reconciliation,3,3,0
3,uniqueness,3,3,0
4,validity,5,5,0


## 5. Export for Power BI

In [18]:
os.makedirs("powerbi/data", exist_ok=True)
run_sql_file("sql/05_export_powerbi.sql")

exports = pd.DataFrame({"file": sorted(os.listdir("powerbi/data"))})
exports["size_kb"] = [round(os.path.getsize(f"powerbi/data/{f}") / 1024, 1) for f in exports["file"]]
exports

2026-10-17 17:34:29,780 | 03_build_warehouse | INFO | Ran sql/05_export_powerbi.sql


,file,size_kb
0,agg_portfolio_monthly.csv,815.4
1,agg_vintage.csv,3.7
2,data_quality_checks.csv,1.8
3,dim_credit_band.csv,0.1
4,dim_month.csv,15.7
5,dim_product.csv,0.2
6,dim_province.csv,0.3
7,fact_renewal.csv,1992.0
8,pd_by_segment.csv,1.3


In [19]:
# Close the connection so other notebooks (and DuckDB tools) can open the warehouse file
con.close()
log.info("Warehouse built and exported")

2026-10-17 17:34:29,936 | 03_build_warehouse | INFO | Warehouse built and exported
